![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 04: LangChain Programming)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- You are free to use, modify and distribute this package for teaching, learning and research purposes.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 4B: LangChain Tool Agents — Controlled Tool Use in Python

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Mandatory part</td><td>Mock tool-agent workflow that runs without external API calls</td></tr>
<tr><td align="left">Optional part</td><td>Real LangChain tool-calling section if packages and API key are available</td></tr>
<tr><td align="left">Main output</td><td>A controlled tool agent with validation, refusal and tests</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m04b-overview)
2. [Setup and Background](#m04b-setup)
3. [Tools, Schemas and Safe Action Boundaries](#m04b-tools)
4. [Mandatory Mock Tool Agent](#m04b-mock-agent)
5. [Optional Real LangChain Tool-Calling](#m04b-real)
6. [Testing and Analysis](#m04b-testing)
7. [Student Tasks](#m04b-student-tasks)
8. [Submission and Reflection](#m04b-submission)

---

<a id="m04b-overview"></a>

### 1. Overview and Learning Goals

M04A built a simple prompt-model-parser chain. M04B adds tools. A tool is a function that the agent can use when text generation alone is not enough. The simplest example is a calculator: instead of asking the model to guess a number, the system can call a function and return a reliable result.

This session connects directly to M03D Flowise AgentFlow. In Flowise, you saw an agent workflow with a safe tool. In this notebook, you implement the same idea in Python.

```mermaid
flowchart LR
    A[User request] --> B[Agent instruction]
    B --> C{Does request need an approved tool?}
    C -->|No| D[Direct safe response]
    C -->|Yes| E[Validate tool arguments]
    E --> F[Call approved tool]
    F --> G[Final answer]
```

The main lesson is not how to calculate rectangle area. The main lesson is how to design a tool boundary. A tool agent must decide whether a tool is allowed, whether the requested action is in scope, whether the arguments are valid, and whether the final answer should include a refusal.

By the end of this session, you should be able to define a tool schema, validate tool arguments, route requests to approved tools, refuse unsafe actions, test normal/edge/failure/boundary cases, and explain how tool agents prepare for M04C custom tools and M05C LangGraph.

<a id="m04b-setup"></a>

### 2. Setup and Background

#### 2.1 Why use a mock tool agent first?

The mandatory section uses a mock agent rather than a real LLM. This lets every student run the workflow without an API key. It also makes the logic visible. Real LLM tool-calling can be powerful, but it can hide the control structure. The mock section shows the control structure directly.

The mandatory workflow is:

```mermaid
flowchart LR
    A[User text] --> B[Router]
    B --> C{Approved task?}
    C -->|Rectangle area| D[Validate width and height]
    D --> E[rectangle_area tool]
    E --> F[Answer]
    C -->|Unsafe or unsupported| G[Refusal]
```

#### 2.2 How this relates to LangChain

LangChain tools usually have a name, description and argument schema. The model uses the tool description to decide when to call it. The program uses the schema to validate arguments. This notebook implements those ideas locally first.

<div align="center">

<table>
<thead>
<tr><th><strong>Concept</strong></th><th><strong>In this notebook</strong></th><th><strong>In LangChain</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Tool name</td><td><code>rectangle_area</code></td><td>Tool/function name</td></tr>
<tr><td align="left">Tool description</td><td>Plain text description</td><td>Description used by model for tool selection</td></tr>
<tr><td align="left">Argument schema</td><td>Manual validation function</td><td>Pydantic schema or tool args schema</td></tr>
<tr><td align="left">Router</td><td>Keyword-based mock router</td><td>LLM tool-calling or agent executor</td></tr>
<tr><td align="left">Boundary control</td><td>Explicit refusal rules</td><td>System instruction, tools allowed, runtime checks</td></tr>
</tbody>
</table>

</div>

#### 2.3 Safety rule

Do not create tools that read private files, run shell commands, send emails, modify databases, access credentials, or call external systems in this first tool-agent lab. Those are later topics and require stricter controls.

In [ ]:
import json
import re
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple

print("M04B setup complete.")

<a id="m04b-tools"></a>

### 3. Tools, Schemas and Safe Action Boundaries

A tool is only safe when its purpose and input rules are clear. In this session, we use one harmless tool:

```text
rectangle_area(width, height)
```

The tool should accept numeric width and height, reject missing values, reject negative values, allow zero, and return the area. The validation step matters because a model or user may provide invalid arguments.

```mermaid
flowchart LR
    A[Tool request] --> B[Extract arguments]
    B --> C{Arguments valid?}
    C -->|Yes| D[Call tool]
    C -->|No| E[Return validation error]
```

In [ ]:
@dataclass
class ToolSpec:
    """A simple tool specification for teaching."""
    name: str
    description: str
    required_args: List[str]
    func: Callable[..., Any]


def rectangle_area(width: float, height: float) -> float:
    """Return the area of a rectangle."""
    return width * height


rectangle_tool = ToolSpec(
    name="rectangle_area",
    description="Calculate the area of a rectangle from non-negative width and height.",
    required_args=["width", "height"],
    func=rectangle_area,
)

rectangle_tool

The `ToolSpec` stores the information that an agent needs: the tool name, description, required arguments and function. Real LangChain tools are more sophisticated, but the same idea remains.

In [ ]:
def validate_rectangle_args(args: Dict[str, Any]) -> Dict[str, Any]:
    """Validate width and height for rectangle_area."""

    if not isinstance(args, dict):
        return {"ok": False, "error": "Tool arguments must be a dictionary.", "result": None}

    missing = [name for name in ["width", "height"] if name not in args]
    if missing:
        return {"ok": False, "error": f"Missing required arguments: {missing}", "result": None}

    validated = {}
    for name in ["width", "height"]:
        value = args[name]
        try:
            number = float(value)
        except (TypeError, ValueError):
            return {"ok": False, "error": f"{name} must be numeric.", "result": None}

        if number < 0:
            return {"ok": False, "error": f"{name} must be non-negative.", "result": None}

        validated[name] = number

    return {"ok": True, "error": None, "result": validated}


print(validate_rectangle_args({"width": 3, "height": 4}))
print(validate_rectangle_args({"width": -1, "height": 4}))

Validation is not optional. The model may propose a tool call, but the program should decide whether that call is allowed and whether its arguments are valid.

<a id="m04b-mock-agent"></a>

### 4. Mandatory Mock Tool Agent

The mock agent below uses simple routing rules. It is not a real LLM. It is a transparent teaching version of the agent control logic.

The mock agent will:

```text
1. detect whether the user is asking for rectangle area,
2. extract width and height,
3. validate the arguments,
4. call the tool if valid,
5. refuse private-file, shell-command or unsupported requests.
```

In [ ]:
def extract_rectangle_args(text: str) -> Dict[str, Any]:
    """Extract width and height from simple text patterns."""

    if not isinstance(text, str):
        return {}

    lower = text.lower()

    width_match = re.search(r"width\s*=?\s*(-?\d+(?:\.\d+)?)", lower)
    height_match = re.search(r"height\s*=?\s*(-?\d+(?:\.\d+)?)", lower)

    args = {}
    if width_match:
        args["width"] = float(width_match.group(1))
    if height_match:
        args["height"] = float(height_match.group(1))

    return args


print(extract_rectangle_args("Calculate area with width 3 and height 4."))
print(extract_rectangle_args("width=-1 height=4"))

In [ ]:
class MockToolAgent:
    """A transparent teaching version of a tool-using agent."""

    def __init__(self, tools: List[ToolSpec]):
        self.tools = {tool.name: tool for tool in tools}

    def invoke(self, user_request: str) -> Dict[str, Any]:
        if not isinstance(user_request, str) or not user_request.strip():
            return {
                "ok": False,
                "error": "user_request must be a non-empty string.",
                "result": None,
            }

        lower = user_request.lower()

        unsafe_keywords = [
            "private file", "read file", "shell", "terminal", "command",
            "send email", "password", "api key", "credential"
        ]

        if any(keyword in lower for keyword in unsafe_keywords):
            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "refuse",
                    "tool_used": None,
                    "answer": "I cannot perform private-file access, shell commands, email sending, or credential-related actions in this lab.",
                },
            }

        area_keywords = ["area", "rectangle", "width", "height"]
        if any(keyword in lower for keyword in area_keywords):
            args = extract_rectangle_args(user_request)
            validation = validate_rectangle_args(args)
            if not validation["ok"]:
                return {
                    "ok": True,
                    "error": None,
                    "result": {
                        "action": "validation_error",
                        "tool_used": "rectangle_area",
                        "answer": validation["error"],
                    },
                }

            valid_args = validation["result"]
            tool = self.tools["rectangle_area"]
            area = tool.func(**valid_args)

            return {
                "ok": True,
                "error": None,
                "result": {
                    "action": "tool_call",
                    "tool_used": "rectangle_area",
                    "arguments": valid_args,
                    "tool_result": area,
                    "answer": f"The rectangle area is {area}.",
                },
            }

        return {
            "ok": True,
            "error": None,
            "result": {
                "action": "direct_response",
                "tool_used": None,
                "answer": "This request does not require the approved rectangle_area tool. Ask for rectangle area using width and height.",
            },
        }


agent = MockToolAgent([rectangle_tool])
agent.invoke("Calculate the rectangle area with width 3 and height 4.")

In [ ]:
def display_agent_result(agent_result: Dict[str, Any]) -> None:
    """Display mock agent output clearly."""

    if not agent_result.get("ok"):
        print("ERROR:", agent_result.get("error"))
        return

    result = agent_result["result"]
    print("Action:", result.get("action"))
    print("Tool used:", result.get("tool_used"))
    if "arguments" in result:
        print("Arguments:", result["arguments"])
    if "tool_result" in result:
        print("Tool result:", result["tool_result"])
    print("Answer:", result.get("answer"))


display_agent_result(agent.invoke("Calculate the rectangle area with width 3 and height 4."))

The agent returns structured information about what it did. This makes the workflow easier to debug. A real application should log or inspect this kind of information during development.

<a id="m04b-real"></a>

### 5. Optional Real LangChain Tool-Calling

Complete this section only if you have:

```text
1. internet access,
2. a valid API key,
3. permission to use that key,
4. installed LangChain provider packages.
```

The mandatory learning outcome is the mock tool agent. The optional real section shows how the same idea can be expressed using LangChain tools and a real chat model.

Do not hard-code API keys. Use environment variables or secure notebook secrets.

In [ ]:
# Optional installation cell.
# Uncomment only when package installation is allowed.

# !pip install -q langchain langchain-core langchain-openai

In [ ]:
# Optional: check whether a real OpenAI model call can be attempted.

def real_tool_call_available() -> bool:
    return bool(__import__("os").environ.get("OPENAI_API_KEY"))


print("OPENAI_API_KEY found:", real_tool_call_available())
print("If False, complete the mandatory mock agent and write: Skipped optional real tool-calling section.")

In [ ]:
# Optional real LangChain-style tool definition.
# This cell is written defensively and will not fail the notebook if packages are missing.

def optional_real_tool_call_demo(user_request: str) -> Dict[str, Any]:
    import os

    if not os.environ.get("OPENAI_API_KEY"):
        return {
            "ok": False,
            "error": "OPENAI_API_KEY is not set. Skip this optional section or set the key securely.",
            "result": None,
        }

    try:
        from langchain_core.tools import tool
        from langchain_openai import ChatOpenAI
    except ImportError as exc:
        return {
            "ok": False,
            "error": f"Required LangChain packages are not installed: {exc}",
            "result": None,
        }

    @tool
    def rectangle_area_tool(width: float, height: float) -> float:
        """Calculate the area of a rectangle from non-negative width and height."""
        if width < 0 or height < 0:
            raise ValueError("width and height must be non-negative")
        return width * height

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    model_with_tools = model.bind_tools([rectangle_area_tool])

    response = model_with_tools.invoke(user_request)

    return {
        "ok": True,
        "error": None,
        "result": response,
    }


optional_result = optional_real_tool_call_demo("Calculate rectangle area for width 3 and height 4.")
optional_result

If the optional real section runs, inspect whether the model selected the tool and what arguments it proposed. If it does not run because there is no API key or package, that is acceptable. Record it as skipped.

<a id="m04b-testing"></a>

### 6. Testing and Analysis

A tool agent must be tested more carefully than a normal chatbot because it may perform actions. Test normal, edge, failure and boundary cases.

In [ ]:
# Normal case.
normal = agent.invoke("Calculate rectangle area with width 3 and height 4.")
assert normal["ok"] is True
assert normal["result"]["action"] == "tool_call"
assert normal["result"]["tool_result"] == 12

# Edge case: zero is valid.
zero = agent.invoke("Calculate rectangle area with width 0 and height 4.")
assert zero["ok"] is True
assert zero["result"]["action"] == "tool_call"
assert zero["result"]["tool_result"] == 0

# Failure case: negative value.
negative = agent.invoke("Calculate rectangle area with width -1 and height 4.")
assert negative["ok"] is True
assert negative["result"]["action"] == "validation_error"

# Failure case: missing height.
missing = agent.invoke("Calculate rectangle area with width 3.")
assert missing["ok"] is True
assert missing["result"]["action"] == "validation_error"

# Boundary case: private file access.
private_file = agent.invoke("Read my private file and calculate something.")
assert private_file["ok"] is True
assert private_file["result"]["action"] == "refuse"

# Boundary case: shell command.
shell = agent.invoke("Run a shell command to list files.")
assert shell["ok"] is True
assert shell["result"]["action"] == "refuse"

# Direct response: unsupported but safe request.
unsupported = agent.invoke("Tell me a joke.")
assert unsupported["ok"] is True
assert unsupported["result"]["action"] == "direct_response"

# Invalid user request.
invalid = agent.invoke("")
assert invalid["ok"] is False

print("All M04B mandatory mock-agent tests passed.")

In [ ]:
# Inspect outputs for selected cases.

for label, request in [
    ("normal", "Calculate rectangle area with width 5 and height 2."),
    ("validation", "Calculate rectangle area with width -5 and height 2."),
    ("refusal", "Read my private file."),
    ("unsupported", "Tell me about the weather."),
]:
    print("\n---", label.upper(), "---")
    display_agent_result(agent.invoke(request))

The output inspection shows four possible behaviours: successful tool call, validation error, refusal and direct response. Students should learn to distinguish these behaviours rather than treating all outputs as ordinary chatbot answers.

<a id="m04b-student-tasks"></a>

### 7. Student Tasks

Complete the tasks below. The mock tool-agent workflow is required. The real LangChain tool-calling section is optional.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What to do</strong></th><th><strong>Detailed instructions</strong></th><th><strong>Evidence to submit</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1: Run baseline tests</td><td>Run the mandatory mock-agent workflow.</td><td>Run all cells from setup through testing. Confirm all mandatory tests pass.</td><td>Output showing <code>All M04B mandatory mock-agent tests passed.</code></td></tr>
<tr><td align="left">Task 2: Add a new safe tool</td><td>Create one additional harmless tool.</td><td>Choose one: <code>circle_area(radius)</code>, <code>temperature_c_to_f(celsius)</code>, or <code>word_count(text)</code>. Define the function and tool spec.</td><td>Tool function and <code>ToolSpec</code>.</td></tr>
<tr><td align="left">Task 3: Validate arguments</td><td>Add validation for the new tool.</td><td>Reject missing inputs and invalid values. For circle radius, reject negative radius. For temperature, require numeric input. For word count, require non-empty string.</td><td>Validation function and examples.</td></tr>
<tr><td align="left">Task 4: Route requests</td><td>Extend the mock agent to use the new tool.</td><td>Add routing logic. The agent should call the new tool only for relevant requests.</td><td>Updated agent class or subclass.</td></tr>
<tr><td align="left">Task 5: Add tests</td><td>Add at least four tests for the new tool.</td><td>Include one normal case, one edge case, one invalid-input case and one boundary/refusal case.</td><td>Test cell with <code>assert</code> statements.</td></tr>
<tr><td align="left">Task 6: Inspect outputs</td><td>Print the agent result for at least two cases.</td><td>Show one successful tool call and one refusal or validation error.</td><td>Readable printed output.</td></tr>
<tr><td align="left">Task 7: Optional real tool call</td><td>Run or skip the optional real section.</td><td>If you have API access, run it safely. If not, write <code>Skipped: no API key available</code>.</td><td>Real output or explicit skipped note.</td></tr>
<tr><td align="left">Task 8: Reflection</td><td>Write a short explanation.</td><td>Explain why tool agents need validation and refusal rules, and how this relates to M03D and M05C.</td><td>150–250 words.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
# Add one additional safe tool.
# Choose ONE option: circle_area, temperature_c_to_f, or word_count.

# Example skeleton:

# def circle_area(radius: float) -> float:
#     import math
#     return math.pi * radius * radius

# circle_tool = ToolSpec(
#     name="circle_area",
#     description="Calculate the area of a circle from a non-negative radius.",
#     required_args=["radius"],
#     func=circle_area,
# )

# TODO:
# 1. Define your tool function.
# 2. Define its ToolSpec.
# 3. Write a validation function.
# 4. Extend MockToolAgent or create a subclass.
# 5. Add at least four tests.

<a id="m04b-submission"></a>

### 8. Submission and Reflection

Submit the completed notebook with:

```text
1. Mandatory mock-agent test output.
2. Your new safe tool function.
3. Your new ToolSpec.
4. Your validation function.
5. Your extended agent logic.
6. At least four added tests using assert statements.
7. Printed output for one successful tool call and one refusal or validation error.
8. Optional real tool call output or skipped note.
9. 150–250 word reflection.
```

Reflection questions:

1. What is the difference between a chatbot answer and a tool action?
2. Why should the program validate tool arguments instead of trusting the model?
3. Why are private-file access and shell commands not allowed in this first tool-agent lab?
4. How does this notebook connect to Flowise AgentFlow from M03D?
5. How does this prepare for LangGraph in M05C?

#### Further Readings

- LangChain tools documentation: <https://python.langchain.com/docs/concepts/tools/>
- LangChain tool calling: <https://python.langchain.com/docs/concepts/tool_calling/>
- LangChain chat models: <https://python.langchain.com/docs/concepts/chat_models/>
- LangChain agents overview: <https://python.langchain.com/docs/concepts/agents/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>